# How confident is a predicted protein structure

**Data kit lab** `alphafold-egfr-plddt` · Discovery · about 20 minutes · beginner

**The question.** How confident is the AlphaFold model of human EGFR, residue by residue? What is the mean pLDDT, and what share of the residues falls in each confidence band?

**What you do.** You read a predicted structure file, pull out one confidence score per residue, summarise the scores and check your summary against the database.

1. Open the model file and read its atom lines.
2. Keep one score per residue.
3. Check your reading against the database's own per-residue file.
4. Compute the mean score.
5. Sort the residues into four confidence bands.
6. See where along the chain the model is confident.
7. Compare your numbers with the ones the AlphaFold database API reports.

**The data.** Entry AF-P00533-F1 (model version 6) of the AlphaFold Protein Structure Database: the predicted structure of human epidermal growth factor receptor (EGFR, UniProt P00533). The database is served by EMBL-EBI with data provided by Google DeepMind. The licence is CC-BY-4.0. Nothing is copied into this notebook: the first cell downloads three files from the database.

**What pLDDT is.** A score from 0 to 100 for each residue. A higher score means the method is more confident in the local structure around that residue. It is a prediction of confidence, not a measured error.

**Where this data lives.** The database serves each model as files and serves the summary numbers through an API. Where a company stores models it downloads or computes itself is not covered by a source this lab has opened.

**How to run.** Run the cells from top to bottom. You need Python 3 and `pandas`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'alphafold-egfr-plddt'
FILES = json.loads(r'''{
 "model": {
  "url": "https://alphafold.ebi.ac.uk/files/AF-P00533-F1-model_v6.pdb",
  "name": "model.pdb",
  "sha256": "8b381730ae157a75f2fbc78043aff7b9e605a30420dfe749a56cd61f12125cb9",
  "bytes": 772334
 },
 "confidence": {
  "url": "https://alphafold.ebi.ac.uk/files/AF-P00533-F1-confidence_v6.json",
  "name": "confidence.json",
  "sha256": "926834e03afd8b377e14f5aaaf0f1a8062d74be4391bbe73f5e19adef665b7f4",
  "bytes": 16994
 },
 "model-cif": {
  "url": "https://alphafold.ebi.ac.uk/files/AF-P00533-F1-model_v6.cif",
  "name": "model-cif.cif",
  "sha256": "485413c9edb379204b45cae0191e4d44aa136d160da78ccbed69a4678f8e392f",
  "bytes": 1127973
 }
}''')
PUBLISHED = json.loads(r'''{
 "mean_plddt": {
  "label": "Mean pLDDT",
  "value": 75.94,
  "tolerance": {
   "abs": 0.005
  }
 },
 "fraction_very_low": {
  "label": "Fraction very low (pLDDT below 50)",
  "value": 0.228,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "fraction_low": {
  "label": "Fraction low (50 to below 70)",
  "value": 0.065,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "fraction_confident": {
  "label": "Fraction confident (70 to below 90)",
  "value": 0.233,
  "tolerance": {
   "abs": 0.0005
  }
 },
 "fraction_very_high": {
  "label": "Fraction very high (90 and above)",
  "value": 0.474,
  "tolerance": {
   "abs": 0.0005
  }
 }
}''')
PUBLISHED_FROM = 'AlphaFold DB prediction API, record AF-P00533-F1, fields globalMetricValue and fractionPlddtVeryLow, fractionPlddtLow, fractionPlddtConfident, fractionPlddtVeryHigh | https://alphafold.ebi.ac.uk/api/prediction/P00533'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Open the model file

The model is a text file in PDB format. Each `ATOM` line is one atom. The file starts with a header: what the model is, the paper to cite and the licence.

In [ ]:
lines = data_file("model").read_text().splitlines()
print("lines in the file:", len(lines))
for line in lines:
    if line.startswith("TITLE") or "LICENCE" in line:
        print(line.rstrip())

atom_lines = [line for line in lines if line.startswith("ATOM")]
print("ATOM lines:", len(atom_lines))
print(atom_lines[0])
print(atom_lines[1])

PDB-format lines are cut by column position, not by spaces. The columns used here:

| columns (Python slice) | what |
|---|---|
| `[12:16]` | atom name (`CA` is the alpha carbon, one per residue) |
| `[17:20]` | residue name |
| `[21]` | chain |
| `[22:26]` | residue number |
| `[60:66]` | B-factor column: in an AlphaFold file it holds the pLDDT |

In [ ]:
import pandas as pd

atoms = pd.DataFrame(
    {
        "atom": [line[12:16].strip() for line in atom_lines],
        "residue": [line[17:20] for line in atom_lines],
        "chain": [line[21] for line in atom_lines],
        "number": [int(line[22:26]) for line in atom_lines],
        "plddt": [float(line[60:66]) for line in atom_lines],
    }
)
print(atoms.head(8).to_string(index=False))
print("chains:", list(atoms["chain"].unique()))

## 2. One score per residue

Look at the first rows: every atom of residue 1 carries the same value. The score belongs to the residue, and the file repeats it on each atom.

Check that this holds for all residues, then keep one row per residue. The `CA` atom is the usual choice because every residue has exactly one.

In [ ]:
values_per_residue = atoms.groupby("number")["plddt"].nunique()
print("largest number of different values inside one residue:", int(values_per_residue.max()))

residues = atoms[atoms["atom"] == "CA"].reset_index(drop=True)
print("residues:", len(residues), "| numbered", int(residues["number"].min()), "to", int(residues["number"].max()))
assert residues["number"].is_unique and int(values_per_residue.max()) == 1

## 3. Check your reading against the database's per-residue file

The database also publishes the scores as a small JSON file: residue number, score and a band letter. If your column reading is right, your scores equal the ones in that file.

In [ ]:
import json

confidence = pd.DataFrame(json.loads(data_file("confidence").read_text()))
print(confidence.head(4).to_string(index=False))

same_numbers = (residues["number"].to_numpy() == confidence["residueNumber"].to_numpy()).all()
same_scores = (residues["plddt"].to_numpy() == confidence["confidenceScore"].to_numpy()).all()
print("same residue numbers:", bool(same_numbers), "| same scores:", bool(same_scores))
assert same_numbers and same_scores

## 4. The mean score

The mean pLDDT is the average over residues: add up one score per residue and divide by the number of residues.

**What goes wrong here.** Averaging the B-factor column over all atom lines looks like the same thing. It is not: large residues have more atoms and count more often.

In [ ]:
mean_plddt = residues["plddt"].mean()
mean_over_atoms = atoms["plddt"].mean()

print("mean over", len(residues), "residues:", round(mean_plddt, 4))
print("mean over", len(atoms), "atom lines: ", round(mean_over_atoms, 4), " <- not the quantity we want")

## 5. Four confidence bands

The database sorts residues into four bands. The cut points are 90, 70 and 50.

| band | score | letter in the confidence file |
|---|---|---|
| very high | 90 and above | H |
| confident | 70 to below 90 | M |
| low | 50 to below 70 | L |
| very low | below 50 | D |

**What goes wrong here.** A score can sit exactly on a cut point. Which band gets it? First look for such residues and see which letter the database gave them.

In [ ]:
on_an_edge = confidence[confidence["confidenceScore"].isin([50.0, 70.0, 90.0])]
print(on_an_edge.to_string(index=False))

Two residues score exactly 90.00 and the database labels both `H`. So the lower edge belongs to the band: 90.00 is very high.

Now sort every residue and compare your bands with the database's letters. Every residue should land on the matching letter.

In [ ]:
def band_of(score):
    if score >= 90:
        return "very high"
    if score >= 70:
        return "confident"
    if score >= 50:
        return "low"
    return "very low"


residues["band"] = residues["plddt"].map(band_of)
residues["letter"] = confidence["confidenceCategory"].to_numpy()
print(pd.crosstab(residues["band"], residues["letter"]).to_string())

In [ ]:
ORDER = ["very low", "low", "confident", "very high"]
counts = residues["band"].value_counts().reindex(ORDER)
fractions = counts / len(residues)

summary = pd.DataFrame({"residues": counts, "fraction": fractions.round(4)})
print(summary.to_string())
print("fractions add up to:", round(float(fractions.sum()), 6))

## 6. Where along the chain is the model confident?

A single average hides where the confident and the uncertain parts are. Split the chain into blocks of 100 residues and look at each block.

In [ ]:
residues["block_start"] = (residues["number"] - 1) // 100 * 100 + 1
by_block = residues.groupby("block_start").agg(
    residues=("plddt", "size"),
    mean_plddt=("plddt", "mean"),
    share_very_low=("band", lambda s: (s == "very low").mean()),
)
print(by_block.round(2).to_string())

run_id = (residues["band"] != residues["band"].shift()).cumsum()
runs = residues.groupby(run_id).agg(first=("number", "min"), last=("number", "max"), band=("band", "first"))
runs["length"] = runs["last"] - runs["first"] + 1
longest = runs.sort_values("length", ascending=False).iloc[0]
print()
print("longest unbroken stretch in one band: residues", int(longest["first"]), "to", int(longest["last"]), "|", longest["band"], "|", int(longest["length"]), "residues")

Residues 101 to 600 score above 89 on average. The last 192 residues, 1019 to 1210, are one unbroken very-low stretch.

A very low score does not say the coordinates there are a poor guess at a fixed shape. The region may have no fixed shape at all, or the method may lack the information to predict it. Either way, do not read the coordinates of that stretch as a structure.

## 7. Check against the database

`PUBLISHED` holds five values copied by code from the AlphaFold database API (record `AF-P00533-F1`). `report` prints your value beside each one and says whether it is inside the tolerance: half a unit in the last digit the API prints.

In [ ]:
RESULTS = {
    "mean_plddt": float(mean_plddt),
    "fraction_very_low": float(fractions["very low"]),
    "fraction_low": float(fractions["low"]),
    "fraction_confident": float(fractions["confident"]),
    "fraction_very_high": float(fractions["very high"]),
}
report(RESULTS)

## 8. Why is the mean off by 0.01?

The four fractions match. The mean does not: you get 75.9499 and the API says 75.94. Rounded to two decimals your value is 75.95.

The model also comes as an mmCIF file, and that file states a global pLDDT of its own. Read that one line.

In [ ]:
cif_lines = data_file("model-cif").read_text().splitlines()
stated = [line for line in cif_lines if line.startswith("_ma_qa_metric_global.metric_value")]
print(stated[0])

print(f"your mean, rounded to two decimals: {mean_plddt:.2f}")
print(f"difference from the value in the file: {abs(round(mean_plddt, 2) - float(stated[0].split()[-1])):.2f}")
print(f"difference from the API value:         {abs(round(mean_plddt, 2) - PUBLISHED['mean_plddt']['value']):.2f}")

The database gives two values for the same number: 75.95 in the model's mmCIF file and 75.94 through the API. Your mean agrees with the file.

Why the API differs is not known. Its description calls the field "Average pLDDT score" and says nothing more. A cut-off instead of rounding would explain this entry, but not the others: when this lab was built, the same comparison was run on the three isoform entries of P00533 (not in this notebook). The API value differed from the mean of the published scores by +0.03, -0.01 and 0.00, and the value in each mmCIF file agreed with the mean every time.

What to take away: this lab does **not** reproduce the API's mean within half a unit of its last digit, and says so. In your own work, quote a summary value with its source and version, and recompute it from the per-residue scores when the last digit matters.

## 9. One more mistake worth making once

Put the two residues that score exactly 90.00 in the lower band and see what happens to the fractions.

In [ ]:
strict_very_high = (residues["plddt"] > 90).mean()
strict_confident = ((residues["plddt"] >= 70) & (residues["plddt"] <= 90)).mean()
print("very high, 90.00 included:", round(float(fractions["very high"]), 4), "| 90.00 excluded:", round(float(strict_very_high), 4))
print("confident, 90.00 excluded:", round(float(fractions["confident"]), 4), "| 90.00 included:", round(float(strict_confident), 4))

Two residues out of 1,210 move each fraction by about 0.002. The published fractions have three decimals, so that is enough to turn a match into a mismatch.

## What the result says

About 47 % of the residues are in the very high band and about 23 % in the very low band. The mean of 75.95 sits between two very different regions and describes neither. Report the band shares, or the per-residue profile, next to any mean.

## Try next

1. Count how often each band occurs in blocks of 50 residues, not 100. Where exactly does the confident part end?
2. The API lists three isoform entries for P00533. Change the file addresses in the first cell to one of them (for example `AF-P00533-2-F1`) and rerun. How far is its mean from the value the API reports for that entry?
3. Compute the median pLDDT. Why is it so much higher than the mean here?